# scPolASeq — PBMC 1k v3 Pipeline Notebook

End-to-end orchestration notebook for running the **scPolASeq** alternative polyadenylation (APA) pipeline on 10x Genomics PBMC 1k v3 data (human, GRCh38).

**Steps covered**
1. Import libraries & configure paths  
2. Download / verify human reference datasets on `/cluster` (GRCh38 genome, GENCODE GTF, PolyASite2 atlas)  
3. Download / verify PBMC 1k v3 FASTQs + 10x barcode whitelist  
4. Quality-check all required inputs  
5. Configure pipeline parameters  
6. Run Nextflow stub validation (fast, no containers)  
7. Run the full 9-stage pipeline  
8. Inspect results — PDUI matrix, APA events, model metrics  

> **Prerequisites**: QNAP NAS mounted at `/cluster` with ~15 GB free, Nextflow ≥ 24.04, Apptainer available in WSL.

## 1. Import Required Libraries

In [ ]:
import os
import sys
import shutil
import subprocess
import urllib.request
import tarfile
import gzip
import hashlib
import textwrap
from pathlib import Path
from datetime import datetime

import pandas as pd

# ── Cluster paths ────────────────────────────────────────────────────────────
CLUSTER        = Path("/cluster")
REF_DIR        = CLUSTER / "datasets/reference/GRCh38"
POLYA_DIR      = CLUSTER / "datasets/polya_atlas"
WL_DIR         = CLUSTER / "datasets/10x_whitelists"
PBMC_FASTQ_DIR = CLUSTER / "datasets/pbmc1k/fastqs"
RESULTS_DIR    = CLUSTER / "projects/scPolASeq_pbmc1k/results"
WORK_DIR       = CLUSTER / "scratch/nextflow_work/scpolaseq_pbmc1k"

# ── Pipeline root (two levels up from this notebook) ─────────────────────────
PIPELINE_DIR = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path(os.getcwd()).parent
print(f"Pipeline dir : {PIPELINE_DIR}")
print(f"Cluster root : {CLUSTER}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Date         : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Load Human Reference Dataset

Download GRCh38 primary assembly (GENCODE 44), the GTF annotation, and the PolyASite 2.0 polyA atlas to `/cluster/datasets/`.  
Each download is skipped if the file already exists.

In [ ]:
def check_cluster():
    """Verify /cluster is mounted and writable."""
    if not CLUSTER.exists():
        raise RuntimeError(
            "/cluster not mounted.\n"
            "In WSL run:  sudo mount -t nfs 192.168.0.224:/cluster /cluster"
        )
    test_file = CLUSTER / ".write_test"
    try:
        test_file.touch(); test_file.unlink()
    except PermissionError:
        raise RuntimeError("/cluster is read-only. Check NFS export permissions.")
    print("✓ /cluster is mounted and writable")
    df = shutil.disk_usage(CLUSTER)
    print(f"  Total: {df.total/1e9:.1f} GB  |  Free: {df.free/1e9:.1f} GB")

check_cluster()

In [ ]:
def download_and_decompress(url: str, dest: Path, label: str):
    """Download a gzipped file and decompress to dest. Skips if dest exists and non-empty."""
    if dest.exists() and dest.stat().st_size > 1000:
        print(f"  [SKIP] {label}  →  {dest}")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  Downloading {label}  ({url.split('/')[-1]}) ...")
    tmp = dest.with_suffix(".tmp.gz")
    urllib.request.urlretrieve(url, tmp)
    print(f"  Decompressing → {dest}")
    with gzip.open(tmp, "rb") as f_in, open(dest, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    tmp.unlink()
    print(f"  ✓ {label}  ({dest.stat().st_size / 1e9:.2f} GB)")

# ── Reference URLs ────────────────────────────────────────────────────────────
GENOME_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/GRCh38.primary_assembly.genome.fa.gz"
GTF_URL    = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.annotation.gtf.gz"
POLYA_URL  = "https://polyasite.unibas.ch/download/atlas/2.0/GRCh38.96/atlas.clusters.2.0.GRCh38.96.tsv.gz"

REF_DIR.mkdir(parents=True, exist_ok=True)
POLYA_DIR.mkdir(parents=True, exist_ok=True)

download_and_decompress(GENOME_URL, REF_DIR / "genome.fa",                  "GRCh38 genome")
download_and_decompress(GTF_URL,    REF_DIR / "genes.gtf",                  "GENCODE v44 GTF")
download_and_decompress(POLYA_URL,  POLYA_DIR / "polyasite2_GRCh38.tsv",    "PolyASite 2.0 atlas")
print("\nReference download step complete.")

## 3. Data Preprocessing — PBMC 1k FASTQs + Barcode Whitelist

Download the 10x Genomics PBMC 1k v3 FASTQs (~5 GB) and the 10x v3 barcode whitelist.  
Also builds the `chrom.sizes` file from the genome index if `samtools` is available.

In [ ]:
PBMC_TAR_URL = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_fastqs.tar"
WL_URL       = "https://cf.10xgenomics.com/supp/cell-exp/whitelists/3M-february-2018.txt.gz"

PBMC_FASTQ_DIR.mkdir(parents=True, exist_ok=True)
WL_DIR.mkdir(parents=True, exist_ok=True)

# ── Barcode whitelist ─────────────────────────────────────────────────────────
wl_file = WL_DIR / "3M-february-2018.txt"
if wl_file.exists() and wl_file.stat().st_size > 1000:
    print(f"  [SKIP] Whitelist  →  {wl_file}")
else:
    print("  Downloading 10x v3 barcode whitelist ...")
    download_and_decompress(WL_URL, wl_file, "10x v3 whitelist")
    lines = sum(1 for _ in open(wl_file))
    print(f"  ✓ Whitelist: {lines:,} barcodes")

# ── PBMC 1k v3 FASTQs ────────────────────────────────────────────────────────
fastqs = list(PBMC_FASTQ_DIR.glob("*.fastq.gz"))
if fastqs:
    print(f"  [SKIP] FASTQs already present ({len(fastqs)} files in {PBMC_FASTQ_DIR})")
else:
    print("  Downloading PBMC 1k v3 FASTQs (~5 GB tar) ...")
    tar_tmp = PBMC_FASTQ_DIR.parent / "pbmc_1k_v3_fastqs.tar"
    urllib.request.urlretrieve(PBMC_TAR_URL, tar_tmp)
    print("  Extracting ...")
    with tarfile.open(tar_tmp) as tf:
        # Strip the top-level directory from tar entries
        members = tf.getmembers()
        for m in members:
            m.name = "/".join(m.name.split("/")[1:])  # strip first path component
        tf.extractall(PBMC_FASTQ_DIR, members=[m for m in members if m.name])
    tar_tmp.unlink()
    fastqs = list(PBMC_FASTQ_DIR.glob("*.fastq.gz"))
    print(f"  ✓ FASTQs: {len(fastqs)} files")

# ── chrom.sizes ───────────────────────────────────────────────────────────────
chrom_sizes = REF_DIR / "chrom.sizes"
if not chrom_sizes.exists() and shutil.which("samtools"):
    print("  Building chrom.sizes with samtools ...")
    subprocess.run(["samtools", "faidx", str(REF_DIR / "genome.fa")], check=True)
    with open(chrom_sizes, "w") as fout:
        subprocess.run(["cut", "-f1,2", str(REF_DIR / "genome.fa.fai")], stdout=fout, check=True)
    print(f"  ✓ chrom.sizes: {sum(1 for _ in open(chrom_sizes))} entries")
    
print("\nPBMC data preprocessing step complete.")

## 4. Quality Control — Input Readiness Check

Verify that every file expected by `conf/test_pbmc1k.config` is present and non-empty before Nextflow is invoked.

In [ ]:
required = {
    "Genome FASTA":       REF_DIR / "genome.fa",
    "GTF annotation":     REF_DIR / "genes.gtf",
    "PolyASite atlas":    POLYA_DIR / "polyasite2_GRCh38.tsv",
    "Barcode whitelist":  WL_DIR / "3M-february-2018.txt",
    "PBMC L001 R1 FASTQ": PBMC_FASTQ_DIR / "pbmc_1k_v3_S1_L001_R1_001.fastq.gz",
    "PBMC L001 R2 FASTQ": PBMC_FASTQ_DIR / "pbmc_1k_v3_S1_L001_R2_001.fastq.gz",
    "PBMC L002 R1 FASTQ": PBMC_FASTQ_DIR / "pbmc_1k_v3_S1_L002_R1_001.fastq.gz",
    "PBMC L002 R2 FASTQ": PBMC_FASTQ_DIR / "pbmc_1k_v3_S1_L002_R2_001.fastq.gz",
    "Samplesheet":        PIPELINE_DIR / "tests/data/samplesheet_pbmc1k.csv",
    "Test config":        PIPELINE_DIR / "conf/test_pbmc1k.config",
    "main.nf":            PIPELINE_DIR / "main.nf",
}

rows = []
all_ok = True
for label, path in required.items():
    exists = path.exists()
    size   = f"{path.stat().st_size / 1e6:.1f} MB" if exists else "—"
    status = "✓" if exists else "✗  MISSING"
    if not exists:
        all_ok = False
    rows.append({"Input": label, "Path": str(path), "Size": size, "Status": status})

df_qc = pd.DataFrame(rows)
print(df_qc.to_string(index=False))
print()
if all_ok:
    print("✓ All required inputs present — ready to run the pipeline.")
else:
    print("✗ Some inputs are missing. Re-run sections 2–3 to download them.")

## 5. Configure Pipeline Parameters

Override any `test_pbmc1k.config` defaults here before launching Nextflow.  
Values are exported as environment variables consumed by `nextflow run`.

In [ ]:
# ── Pipeline parameters (override test_pbmc1k.config defaults) ───────────────
PIPELINE_PARAMS = {
    # I/O
    "outdir":              str(RESULTS_DIR),
    # APA detection
    "apa_min_coverage":    "5",
    "apa_min_pdui_delta":  "0.2",
    "apa_grouping_levels": "cluster,cell_type",
    "apa_group_level":     "cluster",
    # ML model
    "apa_model_type":      "random_forest",
    "enable_single_cell_apa_projection": "false",
    # Resources
    "max_cpus":    "8",
    "max_memory":  "10 GB",
    "max_time":    "12 h",
}

CLUSTER_CONFIG   = "/mnt/c/Users/megar/Documents/Megas/Hackaton/cluster_config/nextflow.config"
NXF_PROFILE      = "test_pbmc1k,local_qnap"
NXF_HOME         = "/cluster/scratch/.nextflow"
NXF_SINGULARITY_CACHEDIR = "/cluster/containers"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(WORK_DIR,    exist_ok=True)

print("Pipeline parameters:")
for k, v in PIPELINE_PARAMS.items():
    print(f"  --{k:<42} {v}")

# Build the Nextflow CLI argument string (used in sections 6 & 7)
PARAM_ARGS = " ".join(f"--{k} '{v}'" for k, v in PIPELINE_PARAMS.items())
print("\nReady.")

## 6. Stub Validation (Dimensionality Reduction Analogy)

Run Nextflow with `-stub-run` — all processes execute their `stub:` blocks (no containers, no real compute).  
This is equivalent to a "dry run" that validates channel wiring across all 9 stages without data.

In [ ]:
def run_nextflow(extra_args: list[str], label: str, work_subdir: str) -> bool:
    """Run nextflow in WSL, streaming output line-by-line. Returns True on success."""
    wsl_pipeline = f"/mnt/c/Users/megar/Documents/Megas/Hackaton/scPolASeq"
    cmd = [
        "wsl", "bash", "-c",
        (
            f"export NXF_HOME={NXF_HOME} "
            f"NXF_SINGULARITY_CACHEDIR={NXF_SINGULARITY_CACHEDIR} && "
            f"cd {wsl_pipeline} && "
            f"nextflow run main.nf "
            f"-profile {NXF_PROFILE} "
            f"-c {CLUSTER_CONFIG} "
            f"-work-dir /cluster/scratch/nextflow_work/{work_subdir} "
            + " ".join(extra_args)
        )
    ]
    print(f"{'─'*70}")
    print(f"  {label}")
    print(f"{'─'*70}")
    result = subprocess.run(cmd, text=True)
    ok = result.returncode == 0
    print(f"\n{'✓ SUCCESS' if ok else '✗ FAILED'} (exit code {result.returncode})")
    return ok

ok_stub = run_nextflow(
    extra_args=[
        "-stub-run",
        f"--outdir /cluster/projects/scPolASeq_stub/results",
        "--max_cpus 4",
        "--max_memory '4 GB'",
    ],
    label="Stub validation — 9-stage wiring check (no containers)",
    work_subdir="scpolaseq_stub",
)

## 7. Full Pipeline Run — PBMC 1k v3

Executes all 9 stages against the real PBMC 1k FASTQs using Apptainer containers (`local_qnap` profile).  
Estimated wall time: **2–4 h** on the notebook (8 CPUs, 10 GB RAM).

> Only run after the stub validation passed and all inputs are confirmed present.

In [ ]:
assert ok_stub, "Stub validation failed — fix errors above before running the full pipeline."

ok_full = run_nextflow(
    extra_args=[PARAM_ARGS],
    label="Full pipeline — PBMC 1k v3 (9-stage APA + ML)",
    work_subdir="scpolaseq_pbmc1k",
)

## 8. Inspect Results — APA Events, PDUI Matrix, Model Metrics

Load the pipeline outputs from `/cluster/projects/scPolASeq_pbmc1k/results` and produce summary tables and plots.

In [ ]:
import glob

def _load(pattern: str, label: str) -> pd.DataFrame | None:
    hits = glob.glob(str(RESULTS_DIR / "**" / pattern), recursive=True)
    if not hits:
        print(f"  [not found] {label}  ({pattern})")
        return None
    df = pd.read_csv(hits[0], sep="\t")
    print(f"  ✓ {label}: {df.shape[0]:,} rows × {df.shape[1]} cols  ({hits[0]})")
    return df

print("Loading pipeline outputs from:", RESULTS_DIR)
print()

df_events   = _load("apa_events.tsv",          "APA events")
df_pdui     = _load("pdui_usage_matrix.tsv",    "PDUI matrix")
df_scored   = _load("scored_apa_events.tsv",    "Scored events")
df_metrics  = _load("model_metrics.tsv",        "Model metrics")
df_features = _load("apa_features.tsv",         "Feature table")

if df_metrics is not None:
    print("\n── Model metrics ────────────────")
    print(df_metrics.to_string(index=False))

if df_events is not None:
    n_sig = int(df_events.get("is_significant", pd.Series([False])).sum())
    print(f"\n── APA events: {len(df_events):,} comparisons, {n_sig} significant ──")

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle("scPolASeq — PBMC 1k v3  |  APA summary", fontsize=13)

    # ── 1. ΔPDUI distribution ─────────────────────────────────────────────────
    if df_events is not None and "delta_pdui" in df_events.columns:
        ax = axes[0]
        df_events["delta_pdui"].hist(bins=40, ax=ax, color="#4C72B0", edgecolor="white")
        ax.axvline(0.2,  color="red", linestyle="--", linewidth=1.2, label="|ΔPDUI|=0.2")
        ax.axvline(-0.2, color="red", linestyle="--", linewidth=1.2)
        ax.set_title("ΔPDUI Distribution"); ax.set_xlabel("ΔPDUI"); ax.set_ylabel("# comparisons")
        ax.legend(fontsize=8)

    # ── 2. APA score distribution ─────────────────────────────────────────────
    if df_scored is not None and "apa_score" in df_scored.columns:
        ax = axes[1]
        df_scored["apa_score"].hist(bins=30, ax=ax, color="#55A868", edgecolor="white")
        ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="threshold=0.5")
        ax.set_title("ML APA Score Distribution"); ax.set_xlabel("APA score"); ax.legend(fontsize=8)

    # ── 3. Feature importances ────────────────────────────────────────────────
    df_imp_files = glob.glob(str(RESULTS_DIR / "**" / "feature_importance.tsv"), recursive=True)
    if df_imp_files:
        df_imp = pd.read_csv(df_imp_files[0], sep="\t")
        ax = axes[2]
        df_imp.sort_values("importance").plot.barh(
            x="feature", y="importance", ax=ax,
            color="#C44E52", legend=False)
        ax.set_title("Feature Importance"); ax.set_xlabel("Importance")

    plt.tight_layout()
    out_png = Path(".") / "scpolaseq_pbmc1k_summary.png"
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f"Figure saved → {out_png}")

except ImportError:
    print("matplotlib not installed — skipping plots. pip install matplotlib")